# A. Environment Setup

In [30]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.6 -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# B. Generate Telemetry Data

In [31]:
"""
Synthetic vehicle telemetry data generator.

Produces a CSV matching the schema required by the assignment brief:
vehicle_id, vehicle_model, timestamp, engine_temp, speed, battery_efficiency, lat, lon.

Deliberately injects severe key skew: 3 "hot" vehicle_ids (malfunctioning/high-frequency
sensors) contribute ~1000x the row volume of a typical vehicle. This skew is what the
PySpark salting step (Part 3.2) is designed to fix.

Reproducible via a fixed random seed.
"""

import numpy as np
import pandas as pd

RNG_SEED = 42
TOTAL_ROWS = 500_000
NUM_VEHICLES = 5_000
NUM_HOT_VEHICLES = 3
HOT_MULTIPLIER = 1000

VEHICLE_MODELS = [
    "Volvo_FH16",
    "Scania_R450",
    "MAN_TGX",
    "DAF_XF",
    "Mercedes_Actros",
    "Iveco_S-Way",
]

rng = np.random.default_rng(RNG_SEED)

# ---------------------------------------------------------------------------
# 1. Assign vehicle_ids to models (roughly even split across 6 models)
# ---------------------------------------------------------------------------
vehicle_ids = [f"VEH{str(i).zfill(6)}" for i in range(NUM_VEHICLES)]
vehicle_model_map = {
    vid: VEHICLE_MODELS[i % len(VEHICLE_MODELS)] for i, vid in enumerate(vehicle_ids)
}

# ---------------------------------------------------------------------------
# 2. Decide row count per vehicle_id: solve for baseline `b` such that
#    (NUM_VEHICLES - NUM_HOT_VEHICLES) * b + NUM_HOT_VEHICLES * (b * HOT_MULTIPLIER) = TOTAL_ROWS
# ---------------------------------------------------------------------------
normal_count = NUM_VEHICLES - NUM_HOT_VEHICLES
denominator = normal_count + NUM_HOT_VEHICLES * HOT_MULTIPLIER
baseline_rows_per_vehicle = round(TOTAL_ROWS / denominator)
hot_rows_per_vehicle = baseline_rows_per_vehicle * HOT_MULTIPLIER

hot_vehicle_ids = set(rng.choice(vehicle_ids, size=NUM_HOT_VEHICLES, replace=False))

row_counts = {}
for vid in vehicle_ids:
    row_counts[vid] = hot_rows_per_vehicle if vid in hot_vehicle_ids else baseline_rows_per_vehicle

actual_total = sum(row_counts.values())
# Spread any rounding remainder across normal vehicles (+/-1 row each) instead of
# dumping it all on one vehicle, which could otherwise drive a count negative.
remainder = TOTAL_ROWS - actual_total
normal_vids = [vid for vid in vehicle_ids if vid not in hot_vehicle_ids]
step = 1 if remainder > 0 else -1
for vid in normal_vids[: abs(remainder)]:
    row_counts[vid] += step

print(f"Baseline rows/vehicle: {baseline_rows_per_vehicle}")
print(f"Hot vehicle rows/vehicle: {hot_rows_per_vehicle}")
print(f"Hot vehicle_ids: {sorted(hot_vehicle_ids)}")
print(f"Sum of row_counts before remainder fix: {actual_total}")
print(f"Rounding remainder ({remainder}) spread across {min(abs(remainder), len(normal_vids))} normal vehicles")

# ---------------------------------------------------------------------------
# 3. Generate rows per vehicle
# ---------------------------------------------------------------------------
START_TS = pd.Timestamp("2024-01-01 00:00:00")
END_TS = pd.Timestamp("2024-01-31 23:59:59")
total_seconds_in_range = int((END_TS - START_TS).total_seconds())

chunks = []
for vid in vehicle_ids:
    n = row_counts[vid]
    model = vehicle_model_map[vid]
    is_hot = vid in hot_vehicle_ids

    # Timestamps: uniformly random offsets within the date range, then sorted.
    # Hot vehicles ping far more often within the same window -> denser timeline,
    # consistent with a "malfunctioning sensor spamming logs" narrative.
    offsets = rng.integers(0, total_seconds_in_range, size=n)
    timestamps = START_TS + pd.to_timedelta(offsets, unit="s")
    timestamps = np.sort(timestamps)

    # Engine temp: normal operating band 70-105C, ~2% chance of an overheat spike per row
    engine_temp = rng.normal(loc=88, scale=7, size=n)
    overheat_mask = rng.random(n) < 0.02
    engine_temp[overheat_mask] += rng.uniform(15, 30, size=overheat_mask.sum())
    engine_temp = np.clip(engine_temp, 60, 140)

    # Speed: highway logistics fleet, mostly 60-100 kph with idle/stop periods
    speed = np.clip(rng.normal(loc=75, scale=25, size=n), 0, 130)

    # Battery efficiency: slow degradation over time + noise
    battery_efficiency = np.clip(rng.normal(loc=88, scale=8, size=n), 30, 100)

    # GPS: global logistics fleet -> spread across plausible land-route latitudes/longitudes
    lat = rng.uniform(-55, 70, size=n)
    lon = rng.uniform(-180, 180, size=n)

    chunks.append(
        pd.DataFrame(
            {
                "vehicle_id": vid,
                "vehicle_model": model,
                "timestamp": timestamps,
                "engine_temp": np.round(engine_temp, 2),
                "speed": np.round(speed, 2),
                "battery_efficiency": np.round(battery_efficiency, 2),
                "lat": np.round(lat, 6),
                "lon": np.round(lon, 6),
            }
        )
    )

df = pd.concat(chunks, ignore_index=True)

# ---------------------------------------------------------------------------
# 3b. Inject invalid engine_temp readings (nulls + out-of-range sensor glitches)
#     so the downstream PySpark null/range filter (Part 3.1) has real invalid
#     rows to demonstrably remove, rather than filtering over already-clean data.
# ---------------------------------------------------------------------------
n_total = len(df)
null_idx = rng.choice(n_total, size=int(n_total * 0.005), replace=False)
remaining_idx = np.setdiff1d(np.arange(n_total), null_idx)
glitch_idx = rng.choice(remaining_idx, size=int(n_total * 0.005), replace=False)

df.loc[null_idx, "engine_temp"] = np.nan
# Sensor glitch values: clearly outside the physically plausible 60-140C range
df.loc[glitch_idx, "engine_temp"] = rng.choice([-999.0, 999.0], size=len(glitch_idx))

print(f"\nInjected {len(null_idx):,} null engine_temp readings")
print(f"Injected {len(glitch_idx):,} out-of-range engine_temp readings (sensor glitches)")

# Shuffle row order so timestamps aren't grouped by vehicle_id in the file
# (mirrors real streaming ingestion where pings from many vehicles interleave).
df = df.sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)

OUTPUT_PATH = "synthetic_fleet_telemetry.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nTotal rows written: {len(df):,}")
print(f"Output file: {OUTPUT_PATH}")


Baseline rows/vehicle: 63
Hot vehicle rows/vehicle: 63000
Hot vehicle_ids: [np.str_('VEH000446'), np.str_('VEH003272'), np.str_('VEH003869')]
Sum of row_counts before remainder fix: 503811
Rounding remainder (-3811) spread across 3811 normal vehicles

Injected 2,500 null engine_temp readings
Injected 2,500 out-of-range engine_temp readings (sensor glitches)

Total rows written: 500,000
Output file: synthetic_fleet_telemetry.csv


# C. Ingest and Average

In [32]:
"""
Part 3.1 - Transformations and Actions
========================================
Ingest the synthetic fleet telemetry dataset and compute average engine
temperature per vehicle_model.

Every transformation below is labeled NARROW or WIDE, and that label is
PROVEN (not just asserted) by inspecting the physical plan via .explain(),
looking for the presence/absence of an Exchange node (Spark's physical-plan
marker for a shuffle).

Databricks-compatible: uses spark.read on a relative/DBFS-style path, no
local-only sandbox paths. On Databricks, swap DATA_PATH for a DBFS path
(e.g. "/content/synthetic_fleet_telemetry.csv") after uploading
the file via the Data > Add Data UI.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType
)

# ---------------------------------------------------------------------------
# 0. Spark session
# ---------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("Part3_1_IngestAndAverage")
    .master("local[*]")  # On Databricks this line is simply omitted/ignored -
                          # the cluster's own master is used instead.
    .config("spark.sql.shuffle.partitions", "8")  # small cluster-friendly default;
                                                    # discussed further in 3.2 (salting/partitioning)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA_PATH = "synthetic_fleet_telemetry.csv"  # same directory as this script;


# ---------------------------------------------------------------------------
# 1. INGEST - explicit schema (avoids an extra job for schema inference,
#    which is itself worth naming: inferSchema=True triggers a full data scan
#    as a separate Spark job before the "real" job even starts).
# ---------------------------------------------------------------------------
schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_model", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("engine_temp", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("battery_efficiency", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# NARROW: spark.read itself is not a "dependency" in the RDD-lineage sense
# (there's no parent RDD yet - it's the source), but it's included here as
# step 0 of the pipeline. Each partition = one (or a range of) input file
# splits; no shuffle occurs to build it.
raw_df = spark.read.csv(DATA_PATH, header=True, schema=schema)

print(f"Initial partition count (read): {raw_df.rdd.getNumPartitions()}")
initial_count = raw_df.count()
print(f"Initial row count: {initial_count:,}")

# ---------------------------------------------------------------------------
# 2. NARROW STEP 1 - column selection (projection)
#    Narrow because: each output partition depends on exactly one input
#    partition. Spark can compute this partition-by-partition with zero
#    data movement between executors/partitions.
# ---------------------------------------------------------------------------
selected_df = raw_df.select("vehicle_id", "vehicle_model", "engine_temp")
print(f"\n[NARROW] select() -> partitions: {selected_df.rdd.getNumPartitions()} "
      f"(unchanged from read - 1:1 partition mapping, no shuffle)")

# ---------------------------------------------------------------------------
# 3. NARROW STEP 2 - type validation / cast
#    Narrow for the same reason as select(): withColumn applying a cast is
#    an element-wise transformation, evaluated independently per row within
#    each partition. No data crosses partition boundaries.
# ---------------------------------------------------------------------------
casted_df = selected_df.withColumn("engine_temp", F.col("engine_temp").cast(DoubleType()))
print(f"[NARROW] withColumn(cast) -> partitions: {casted_df.rdd.getNumPartitions()} "
      f"(unchanged - element-wise cast, no shuffle)")

# ---------------------------------------------------------------------------
# 4. NARROW STEP 3 - filter out null/invalid engine_temp readings
#    Narrow because: filter() evaluates a predicate row-by-row within each
#    partition independently. A partition's output rows are a subset of
#    that same partition's input rows - no row ever needs to move to a
#    different partition to be filtered. Output partition count == input
#    partition count (though some partitions may now hold fewer rows).
# ---------------------------------------------------------------------------
# "Invalid" defined as: null, or outside a physically plausible engine temp
# range for a combustion engine (60-140C, matching the generator's clip range).
filtered_df = casted_df.filter(
    F.col("engine_temp").isNotNull()
    & (F.col("engine_temp") >= 60)
    & (F.col("engine_temp") <= 140)
)
print(f"[NARROW] filter() -> partitions: {filtered_df.rdd.getNumPartitions()} "
      f"(unchanged - row-level predicate, no shuffle)")

filtered_count = filtered_df.count()
print(f"\nRow count after null/range filter: {filtered_count:,} "
      f"(dropped {initial_count - filtered_count:,} invalid rows)")

# ---------------------------------------------------------------------------
# 5. WIDE STEP #1 - groupBy(vehicle_model).avg(engine_temp)
#    WIDE because: rows sharing the same vehicle_model can live on ANY
#    partition after the narrow steps above (partitioning so far has been
#    inherited from file splits, not from vehicle_model). To compute one
#    average per vehicle_model, Spark must physically move rows so that
#    all rows for a given vehicle_model land on the same partition before
#    reducing - that data movement across the network/disk between
#    executors IS the shuffle, and it forces a new stage boundary.
#    Physical-plan marker: Exchange hashpartitioning(vehicle_model, N).
#
#    WIDE STEP #2 - orderBy(vehicle_model)
#    Also wide, but for a DIFFERENT reason and via a DIFFERENT partitioning
#    scheme. A global sort needs every partition to know the overall key
#    range so rows land in the correct partition in sorted order (e.g. all
#    "A*" models before all "B*" models across ALL partitions, not just
#    within one). Spark achieves this by sampling the data to estimate key
#    range boundaries, then shuffling with RANGE partitioning (not hash)
#    so each output partition holds a contiguous slice of the sort key.
#    Physical-plan marker: a SECOND, separate Exchange - rangepartitioning
#    (vehicle_model ASC), distinct from the groupBy's hashpartitioning
#    Exchange above. Two wide operations chained back-to-back here means
#    TWO Exchange nodes and two shuffle-boundary stages in the DAG, not one.
# ---------------------------------------------------------------------------
avg_temp_by_model = (
    filtered_df
    .groupBy("vehicle_model")
    .agg(F.avg("engine_temp").alias("avg_engine_temp"))
    .orderBy("vehicle_model")
)

print(f"\n[WIDE x2] groupBy().agg(avg()).orderBy() -> final partitions: "
      f"{avg_temp_by_model.rdd.getNumPartitions()}. Two independent shuffles occurred "
      f"to get here: (1) hash-partition shuffle for the groupBy aggregation, "
      f"(2) range-partition shuffle for the orderBy global sort. Adaptive Query "
      f"Execution (AQE) then coalesced the tiny post-shuffle output (6 rows) down "
      f"to 1 final partition - see the explain() output below for both Exchange nodes.")

# ---------------------------------------------------------------------------
# 6. PROOF #1: .explain() on the physical plan - look for the Exchange node
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("PHYSICAL PLAN for the NARROW-ONLY DataFrame (filtered_df) - expect NO Exchange")
print("=" * 80)
filtered_df.explain(mode="formatted")

print("\n" + "=" * 80)
print("PHYSICAL PLAN for the FULL PIPELINE incl. groupBy - expect Exchange present")
print("=" * 80)
avg_temp_by_model.explain(mode="formatted")

# ---------------------------------------------------------------------------
# 7. PROOF #2: RDD lineage via toDebugString() - shuffle boundary visible as
#    a new "Stage" indentation level / ShuffledRowRDD in the debug string.
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RDD LINEAGE (toDebugString) for narrow-only filtered_df:")
print("=" * 80)
print(filtered_df.rdd.toDebugString().decode("utf-8"))

print("\n" + "=" * 80)
print("RDD LINEAGE (toDebugString) for full pipeline incl. groupBy:")
print("=" * 80)
print(avg_temp_by_model.rdd.toDebugString().decode("utf-8"))

# ---------------------------------------------------------------------------
# 8. ACTION - trigger execution and show the real result
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RESULT: average engine temperature per vehicle_model")
print("=" * 80)
avg_temp_by_model.show(truncate=False)

results_collected = avg_temp_by_model.collect()
print("Collected rows:")
for row in results_collected:
    print(f"  {row['vehicle_model']}: {row['avg_engine_temp']:.2f} C")

spark.stop()


Initial partition count (read): 2
Initial row count: 500,000

[NARROW] select() -> partitions: 2 (unchanged from read - 1:1 partition mapping, no shuffle)
[NARROW] withColumn(cast) -> partitions: 2 (unchanged - element-wise cast, no shuffle)
[NARROW] filter() -> partitions: 2 (unchanged - row-level predicate, no shuffle)

Row count after null/range filter: 495,000 (dropped 5,000 invalid rows)

[WIDE x2] groupBy().agg(avg()).orderBy() -> final partitions: 1. Two independent shuffles occurred to get here: (1) hash-partition shuffle for the groupBy aggregation, (2) range-partition shuffle for the orderBy global sort. Adaptive Query Execution (AQE) then coalesced the tiny post-shuffle output (6 rows) down to 1 final partition - see the explain() output below for both Exchange nodes.

PHYSICAL PLAN for the NARROW-ONLY DataFrame (filtered_df) - expect NO Exchange
== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [3]: [vehicle_id#12688, vehicle_model#12689, engine_temp#

# D. Salting and Skew

In [33]:
"""
Part 3.2 - Optimization: Salting to Mitigate Data Skew
=========================================================

SCOPING CORRECTION vs Part 3.1:
Part 3.1 aggregated by vehicle_model (only 6 distinct values, assigned
round-robin across 5,000 vehicles) - that key has NO real skew; every
model ends up with a roughly similar row count by construction.

The actual ~1000x skew the assignment describes ("some specific trucks
generate ~1000x more logs than others") lives at the vehicle_id level: 3
hot vehicle_ids out of 5,000 each carry ~63,000 rows vs ~62-63 rows for
everyone else. So THIS section re-scopes the aggregation to
"average engine temperature per vehicle_id" - that is the key that is
actually skewed, and therefore the only key for which a salting demo
means anything. Aggregating by vehicle_model here would prove nothing.

Databricks-compatible: no local-only paths; swap DATA_PATH for a DBFS
path on Databricks as in Part 3.1.
"""

import math
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# ---------------------------------------------------------------------------
# 0. Spark session
# ---------------------------------------------------------------------------
# Partitioning strategy note (tied back to Part 3.1's spark.sql.shuffle.partitions=8):
# 3.1 grouped by vehicle_model - only 6 distinct keys, so 8 shuffle partitions
# was already more than enough. Here we group (and salt) by vehicle_id, which
# after salting produces up to NUM_VEHICLES * NUM_SALT_BUCKETS = 5,000 * 10 =
# 50,000 distinct salted keys. With only 8 output partitions, the pigeonhole
# principle means several of a hot vehicle's 10 salted sub-keys would still
# collide back onto the same partition by chance, partially undoing the
# salting benefit. We raise shuffle.partitions to 20 here - still HASH
# partitioning (justified below), just recalibrated for this key's higher
# cardinality. This is a deliberate, explained change from 3.1, not an
# inconsistency.
NUM_SALT_BUCKETS = 10
SHUFFLE_PARTITIONS = 20

spark = (
    SparkSession.builder
    .appName("Part3_2_SaltingSkew")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

DATA_PATH = "/content/synthetic_fleet_telemetry.csv"

schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_model", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("engine_temp", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("battery_efficiency", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# ---------------------------------------------------------------------------
# 1. Ingest + narrow cleaning (same steps as Part 3.1, condensed here since
#    the narrow-vs-wide proof for these specific steps was already
#    established there; re-derivation not repeated).
# ---------------------------------------------------------------------------
raw_df = spark.read.csv(DATA_PATH, header=True, schema=schema)
filtered_df = (
    raw_df
    .select("vehicle_id", "engine_temp")
    .filter(F.col("engine_temp").isNotNull() & (F.col("engine_temp") >= 60) & (F.col("engine_temp") <= 140))
)
filtered_df.cache()
total_rows = filtered_df.count()
print(f"Total valid rows after cleaning: {total_rows:,}")

# ---------------------------------------------------------------------------
# 2. BEFORE EVIDENCE - prove the skew exists, don't just assert it
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2: BEFORE evidence - row count per vehicle_id (top/bottom)")
print("=" * 80)

rows_per_vehicle = filtered_df.groupBy("vehicle_id").count().cache()
rows_per_vehicle_sorted_desc = rows_per_vehicle.orderBy(F.desc("count"))
rows_per_vehicle_sorted_asc = rows_per_vehicle.orderBy(F.asc("count"))

print("Top 5 vehicle_ids by row count (expected: the 3 hot ones + 2 normal for contrast):")
rows_per_vehicle_sorted_desc.show(5, truncate=False)

print("Bottom 5 vehicle_ids by row count:")
rows_per_vehicle_sorted_asc.show(5, truncate=False)

stats = rows_per_vehicle.agg(
    F.min("count").alias("min_count"),
    F.max("count").alias("max_count"),
    F.avg("count").alias("avg_count"),
).collect()[0]
skew_ratio = stats["max_count"] / stats["avg_count"]
print(f"\nmin rows/vehicle: {stats['min_count']}, max: {stats['max_count']}, "
      f"avg: {stats['avg_count']:.2f} -> max/avg skew ratio: {skew_ratio:.1f}x")

# ---------------------------------------------------------------------------
# 2b. BEFORE EVIDENCE, at the PARTITION level (not just per-key counts).
#
# Why repartition-by-key specifically (rather than reading the skew off the
# groupBy().avg() plan directly): Spark's Catalyst optimizer automatically
# performs a map-side PARTIAL aggregation for associative/commutative
# functions like sum/count/avg BEFORE the shuffle (visible in Part 3.1's
# explain() output as the first HashAggregate, ahead of the Exchange). That
# partial aggregation already collapses each partition's rows for a given
# key down to a single (sum, count) pair before anything moves across the
# network - so for a plain avg(), the shuffle-stage data volume is only
# mildly sensitive to raw key skew.
#
# The scenario where raw key skew directly and severely imbalances
# partition sizes is a RAW repartition by key - e.g. what you would need
# before a non-associative per-vehicle computation (a UDF, a window
# function, or the per-vehicle predictive-maintenance modeling this
# telemetry is ultimately destined for). That is what we demonstrate here:
# forcing all of one vehicle_id's rows onto one partition, unsalted.
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 2b: BEFORE evidence - partition sizes after repartition(vehicle_id), unsalted")
print("=" * 80)

before_partition_sizes = (
    filtered_df
    .repartition(SHUFFLE_PARTITIONS, "vehicle_id")
    .rdd.glom().map(len).collect()
)
for i, size in enumerate(before_partition_sizes):
    marker = "  <-- HOT" if size > 5 * (total_rows / SHUFFLE_PARTITIONS) else ""
    print(f"  Partition {i:2d}: {size:>7,} rows{marker}")
print(f"\nExpected even share per partition (no skew case): ~{total_rows // SHUFFLE_PARTITIONS:,} rows")
print(f"Actual max partition size: {max(before_partition_sizes):,} "
      f"({max(before_partition_sizes) / (total_rows / SHUFFLE_PARTITIONS):.1f}x the even-share size)")

# ---------------------------------------------------------------------------
# 3. SALTING IMPLEMENTATION
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 3: Salting - stage 1 (salted partial aggregation)")
print("=" * 80)

# Append a random suffix (0..NUM_SALT_BUCKETS-1) to vehicle_id. This spreads
# any single hot vehicle_id's rows across NUM_SALT_BUCKETS distinct keys
# instead of one, so no single downstream partition can be dominated by one
# vehicle_id's raw rows.
salted_df = filtered_df.withColumn(
    "salted_key",
    F.concat(F.col("vehicle_id"), F.lit("_"), (F.floor(F.rand(seed=42) * NUM_SALT_BUCKETS)).cast("string"))
)

# CRITICAL CORRECTNESS POINT: this is an AVERAGE, not a SUM/COUNT.
# We must NOT average the per-salt-bucket averages together - salt buckets
# will have uneven row counts (a hot vehicle's ~6,300 rows/bucket vs a
# normal vehicle's ~6 rows/bucket), so a naive avg-of-avgs would silently
# give every bucket equal weight regardless of how many rows backed it,
# producing a mathematically wrong result. Instead we carry forward
# (sum, count) per salted key - both of which ARE safe to add across
# buckets - and defer the division until the final de-salted step.
stage1_df = salted_df.groupBy("salted_key").agg(
    F.sum("engine_temp").alias("partial_sum"),
    F.count("engine_temp").alias("partial_count"),
)

print(f"Stage 1 output rows (distinct salted keys): {stage1_df.count():,} "
      f"(<= {5000 * NUM_SALT_BUCKETS:,} possible combinations)")
print("Sample of stage 1 output (salted keys, partial sum/count):")
stage1_df.orderBy("salted_key").show(5, truncate=False)

print("\n" + "=" * 80)
print("STEP 3: Salting - stage 2 (strip salt suffix, weighted recombination)")
print("=" * 80)

# Strip the salt suffix to recover the true vehicle_id. vehicle_id itself
# never contains an underscore (format "VEHxxxxxx"), so splitting on "_"
# and taking element 0 is a safe, unambiguous inverse of the concat above.
stage2_df = stage1_df.withColumn(
    "vehicle_id", F.split(F.col("salted_key"), "_").getItem(0)
)

# Weighted recombination: sum(sum) / sum(count), grouped on the REAL
# vehicle_id - NOT avg(partial_sum / partial_count), which would again be
# an unweighted average-of-averages and wrong for the same reason as above.
final_avg_df = (
    stage2_df
    .groupBy("vehicle_id")
    .agg(
        F.sum("partial_sum").alias("total_sum"),
        F.sum("partial_count").alias("total_count"),
    )
    .withColumn("avg_engine_temp", F.col("total_sum") / F.col("total_count"))
    .select("vehicle_id", "avg_engine_temp", "total_count")
)
final_avg_df.cache()

print(f"Final de-salted result rows (should equal vehicle_id cardinality): {final_avg_df.count():,}")
final_avg_df.orderBy("vehicle_id").show(5, truncate=False)

print("\n" + "=" * 80)
print("STEP 3: explain() for the full two-stage salted aggregation")
print("=" * 80)
final_avg_df.explain(mode="formatted")

# ---------------------------------------------------------------------------
# 4. CORRECTNESS SPOT-CHECK - salted+recombined result vs plain unsalted groupBy
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 4: Correctness spot-check (salted result vs plain unsalted groupBy)")
print("=" * 80)

baseline_avg_df = filtered_df.groupBy("vehicle_id").agg(F.avg("engine_temp").alias("avg_engine_temp"))

hot_ids = [r["vehicle_id"] for r in rows_per_vehicle_sorted_desc.limit(3).collect()]
normal_ids = [r["vehicle_id"] for r in rows_per_vehicle_sorted_asc.limit(2).collect()]
check_ids = hot_ids + normal_ids

baseline_map = {
    r["vehicle_id"]: r["avg_engine_temp"]
    for r in baseline_avg_df.filter(F.col("vehicle_id").isin(check_ids)).collect()
}
salted_map = {
    r["vehicle_id"]: r["avg_engine_temp"]
    for r in final_avg_df.filter(F.col("vehicle_id").isin(check_ids)).collect()
}

print(f"{'vehicle_id':<12}{'unsalted avg':<16}{'salted+recombined avg':<24}{'abs diff':<12}{'match?'}")
for vid in check_ids:
    b, s = baseline_map[vid], salted_map[vid]
    diff = abs(b - s)
    match = math.isclose(b, s, rel_tol=1e-9)
    tag = "hot" if vid in hot_ids else "normal"
    print(f"{vid:<12}{b:<16.6f}{s:<24.6f}{diff:<12.2e}{match}  ({tag})")

# ---------------------------------------------------------------------------
# 5. AFTER EVIDENCE - partition sizes after repartitioning by the SALTED key
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 5: AFTER evidence - partition sizes after repartition(salted_key)")
print("=" * 80)

after_partition_sizes = (
    salted_df
    .repartition(SHUFFLE_PARTITIONS, "salted_key")
    .rdd.glom().map(len).collect()
)
for i, size in enumerate(after_partition_sizes):
    marker = "  <-- still hot" if size > 5 * (total_rows / SHUFFLE_PARTITIONS) else ""
    print(f"  Partition {i:2d}: {size:>7,} rows{marker}")

before_mean = total_rows / SHUFFLE_PARTITIONS
before_stdev = (sum((x - before_mean) ** 2 for x in before_partition_sizes) / len(before_partition_sizes)) ** 0.5
after_mean = total_rows / SHUFFLE_PARTITIONS
after_stdev = (sum((x - after_mean) ** 2 for x in after_partition_sizes) / len(after_partition_sizes)) ** 0.5

print(f"\nBEFORE - max partition: {max(before_partition_sizes):,} rows (stdev across partitions: {before_stdev:.0f})")
print(f"AFTER  - max partition: {max(after_partition_sizes):,} rows (stdev across partitions: {after_stdev:.0f})")
print(f"\nMax-partition reduction factor: {max(before_partition_sizes) / max(after_partition_sizes):.1f}x smaller")

# ---------------------------------------------------------------------------
# 6. Partitioning strategy justification: HASH vs RANGE
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 6: Partitioning strategy - hash vs range (see explain() above for proof)")
print("=" * 80)
print(f"""
Both stage-1 and stage-2 groupBy's above use HASH partitioning by default
(visible in the explain() output as "hashpartitioning(salted_key, {SHUFFLE_PARTITIONS})"
and "hashpartitioning(vehicle_id, {SHUFFLE_PARTITIONS})" respectively) - this is
Spark's automatic choice for groupBy/aggregation, and it is the right one here:

- HASH partitioning: P = hash(key) mod N. O(1) to compute per row, and -
  critically - once the key space is roughly uniform (which is exactly what
  salting engineers for), hash partitioning spreads keys evenly across all
  N partitions with no extra coordination. It has no notion of key ORDER,
  which is fine because nothing downstream needs vehicle_ids or salted keys
  in sorted order.

- RANGE partitioning (contrast: Part 3.1's orderBy(vehicle_model) used this
  - "rangepartitioning(vehicle_model ASC, 8)" in that script's explain output)
  requires sampling the data to estimate key-range boundaries so each
  partition holds a contiguous slice of the sort key. That coordination cost
  only pays for itself when a GLOBAL ORDER is actually needed downstream
  (range scans, sorted output, ASOF-style joins). Grouping/aggregating by
  vehicle_id has no such requirement - we only need "same key -> same
  partition", not "partition N holds a contiguous key range" - so range
  partitioning would add sampling overhead here for zero benefit.

vehicle_id (and the salted key derived from it) is a high-cardinality,
non-ordinal identifier - there is no meaningful "less than"/"greater than"
between VEH003272 and VEH000446 that any query in this pipeline cares
about. That absence of a meaningful order is precisely the condition under
which hash partitioning is the correct choice over range partitioning.
""")

spark.stop()

Total valid rows after cleaning: 495,000

STEP 2: BEFORE evidence - row count per vehicle_id (top/bottom)
Top 5 vehicle_ids by row count (expected: the 3 hot ones + 2 normal for contrast):
+----------+-----+
|vehicle_id|count|
+----------+-----+
|VEH003869 |62387|
|VEH000446 |62378|
|VEH003272 |62375|
|VEH004933 |63   |
|VEH004270 |63   |
+----------+-----+
only showing top 5 rows

Bottom 5 vehicle_ids by row count:
+----------+-----+
|vehicle_id|count|
+----------+-----+
|VEH000352 |58   |
|VEH002304 |58   |
|VEH001625 |58   |
|VEH001559 |58   |
|VEH000507 |58   |
+----------+-----+
only showing top 5 rows


min rows/vehicle: 58, max: 62387, avg: 99.00 -> max/avg skew ratio: 630.2x

STEP 2b: BEFORE evidence - partition sizes after repartition(vehicle_id), unsalted
  Partition  0:  16,145 rows
  Partition  1:  15,468 rows
  Partition  2:  77,597 rows
  Partition  3:  13,793 rows
  Partition  4:  15,575 rows
  Partition  5:  14,980 rows
  Partition  6:  16,642 rows
  Partition  7:  16,4

# E. Fault Tolerance

In [34]:
"""
Part 3.3 - Fault Tolerance via RDD Lineage (NOT Replication)
==============================================================

Core claim to prove empirically, not just assert: a lost/failed partition is
recovered by RE-EXECUTING THE LINEAGE (the recorded chain of transformations)
against the data still durably available at the source, NOT by reading a
replica copy of that partition's output that Spark kept around "just in
case". Spark keeps zero extra copies of intermediate RDD data by default -
recomputation from lineage is the only mechanism, and this script
deliberately crashes one partition's first task attempt to prove that
recomputation is real, not theoretical.

Databricks-compatible: no local-only paths.
"""

import os
from pyspark import TaskContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# TARGET_PARTITION (the partition whose FIRST attempt we deliberately crash) is
# chosen dynamically below, once the real partition count is known - the
# actual count depends on the cluster config/environment (we observed 10 and
# 4 across different runs already), so a hardcoded literal would go out of
# range if that count changes again.

# ---------------------------------------------------------------------------
# 0. Spark session.
#
# EMPIRICAL FINDING (verified by actually running this with master="local[*]"
# first): even with spark.task.maxFailures=4 explicitly configured, a task
# that failed once caused the WHOLE JOB TO ABORT immediately - the real log
# line was:
#   "ERROR TaskSetManager: Task 3 in stage 1.0 failed 1 times; aborting job"
# This is a documented Spark quirk, not a bug in this script: for `local` /
# `local[*]` masters, Spark's SparkContext hardcodes the effective
# maxTaskFailures to 1 internally, regardless of what spark.task.maxFailures
# is set to. local[*] therefore CANNOT be used to demonstrate retry-based
# recovery - it always aborts on the first failure.
#
# `local-cluster[N, cores, memory]` mode does NOT have this override: it
# spawns genuine separate Worker/Executor JVM processes (still on this one
# machine, but architecturally a real distributed scheduler, not the
# same-process shortcut local[*] takes). It honors spark.task.maxFailures
# properly, which is what actually lets us prove the retry-and-recover
# mechanism below - and is the behavior that matches a real Databricks
# cluster, not local[*]'s.
# ---------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("Part3_3_FaultTolerance")
    .master("local-cluster[2,2,1024]")  # 2 worker JVMs, 2 cores/1GB each - see note above
    .config("spark.sql.shuffle.partitions", "20")
    .config("spark.task.maxFailures", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

configured_max_failures = spark.sparkContext.getConf().get("spark.task.maxFailures")
print(f"spark.task.maxFailures = {configured_max_failures} (must be >1 for retry-on-failure to occur)")

DATA_PATH = "synthetic_fleet_telemetry.csv"
schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_model", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("engine_temp", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("battery_efficiency", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# ---------------------------------------------------------------------------
# 1. Ingest + narrow cleaning - same as Part 3.1/3.2, not re-derived here.
# ---------------------------------------------------------------------------
raw_df = spark.read.csv(DATA_PATH, header=True, schema=schema)
filtered_df = (
    raw_df.select("vehicle_id", "engine_temp")
    .filter(F.col("engine_temp").isNotNull() & (F.col("engine_temp") >= 60) & (F.col("engine_temp") <= 140))
)

num_partitions = filtered_df.rdd.getNumPartitions()
TARGET_PARTITION = num_partitions // 2  # derived from the actual partition count, not hardcoded
print(f"filtered_df partitions: {num_partitions} -> TARGET_PARTITION chosen as {TARGET_PARTITION} "
      f"(middle partition, computed from the real count above)")

# ---------------------------------------------------------------------------
# 2. Lineage size vs. actual data size - make "lineage is cheap, data is
#    expensive" measurable rather than asserted.
# ---------------------------------------------------------------------------
lineage_str = filtered_df.rdd.toDebugString().decode("utf-8")
lineage_bytes = len(lineage_str.encode("utf-8"))
lineage_lines = lineage_str.count("\n") + 1
print(f"\nLineage description: {lineage_lines} lines / {lineage_bytes} bytes total - "
      f"this fixed-size 'recipe' is ALL Spark needs to keep to recompute any partition.")

csv_file_size_bytes = os.path.getsize(DATA_PATH)
RAW_TOTAL_ROWS = 500_000
bytes_per_row_estimate = csv_file_size_bytes / RAW_TOTAL_ROWS
print(f"Source CSV: {csv_file_size_bytes:,} bytes / {RAW_TOTAL_ROWS:,} rows "
      f"-> ~{bytes_per_row_estimate:.1f} bytes/row")

# ---------------------------------------------------------------------------
# 3. Baseline (no failure) row count per partition, for later spot-check.
# ---------------------------------------------------------------------------
def count_partition(index, iterator):
    yield (index, sum(1 for _ in iterator))

baseline_counts = dict(filtered_df.rdd.mapPartitionsWithIndex(count_partition).collect())
print(f"\n[BASELINE - no failure injected] partition {TARGET_PARTITION} row count: "
      f"{baseline_counts[TARGET_PARTITION]:,}")
print(f"[BASELINE] total row count across all {num_partitions} partitions: "
      f"{sum(baseline_counts.values()):,}")

estimated_partition_bytes = baseline_counts[TARGET_PARTITION] * bytes_per_row_estimate
print(f"Estimated data volume Spark must re-read/recompute if partition {TARGET_PARTITION} "
      f"is lost: ~{estimated_partition_bytes:,.0f} bytes (~{estimated_partition_bytes / 1024:.1f} KB)")
print(f"Lineage-to-data leverage: {lineage_bytes} bytes of lineage metadata is enough to "
      f"regenerate ~{estimated_partition_bytes / 1024:.1f} KB of actual data - a "
      f"{estimated_partition_bytes / lineage_bytes:,.0f}x ratio. Spark stores the "
      f"{lineage_bytes}-byte recipe, not a redundant copy of the "
      f"{estimated_partition_bytes / 1024:.1f} KB it produces.")

# ---------------------------------------------------------------------------
# 4. THE ACTUAL FAILURE INJECTION AND RECOVERY.
#    attemptNumber() == 0 -> deliberately raise, simulating a crashed task.
#    Spark's scheduler (per spark.task.maxFailures) reschedules the SAME
#    partition as attempt 1, recomputing it from lineage (re-read + re-filter
#    that one file split) - not from any cached/replicated copy, because none
#    exists.
# ---------------------------------------------------------------------------
def inject_failure_and_count(index, iterator):
    ctx = TaskContext.get()
    attempt = ctx.attemptNumber()
    rows = list(iterator)
    row_count = len(rows)
    if index == TARGET_PARTITION and attempt == 0:
        raise RuntimeError(
            f"INJECTED FAILURE: simulated crash on partition {index}, attempt {attempt} "
            f"(this partition holds {row_count} rows - none are lost, they will be "
            f"recomputed from source on retry, not read from a replica)"
        )
    yield (index, row_count, attempt)

print("\n" + "=" * 80)
print(f"STEP 4: Triggering job with a deliberate crash on partition {TARGET_PARTITION}, attempt 0.")
print(f"Watch for a WARN 'Lost task {TARGET_PARTITION}.0 ...' line below, followed by the "
      f"job completing anyway via a successful retry (attempt 1).")
print("=" * 80)

result_with_recovery = filtered_df.rdd.mapPartitionsWithIndex(inject_failure_and_count).collect()

print("\nJob completed. Per-partition (index, row_count, attempt_that_succeeded):")
for idx, cnt, attempt in sorted(result_with_recovery):
    marker = "  <-- this is the partition we crashed on attempt 0" if idx == TARGET_PARTITION else ""
    print(f"  partition {idx}: {cnt:,} rows, succeeded on attempt {attempt}{marker}")

recovered_row = next(r for r in result_with_recovery if r[0] == TARGET_PARTITION)
recovered_partition_count, recovered_attempt = recovered_row[1], recovered_row[2]

print(f"\nSPOT-CHECK: partition {TARGET_PARTITION} baseline row count = "
      f"{baseline_counts[TARGET_PARTITION]:,}, recovered row count = {recovered_partition_count:,} -> "
      f"{'MATCH (recomputation was correct and complete)' if recovered_partition_count == baseline_counts[TARGET_PARTITION] else 'MISMATCH - investigate'}")
print(f"Partition {TARGET_PARTITION} succeeded on attempt {recovered_attempt} "
      f"({'retry occurred as expected' if recovered_attempt > 0 else 'NO retry occurred - see honesty note below'})")

total_after_recovery = sum(cnt for _, cnt, _ in result_with_recovery)
baseline_total = sum(baseline_counts.values())
print(f"\nTotal rows across all partitions after recovery: {total_after_recovery:,} "
      f"(baseline total was {baseline_total:,}) -> "
      f"{'MATCH' if total_after_recovery == baseline_total else 'MISMATCH'}")

if recovered_attempt == 0:
    print("\nHONESTY NOTE: attempt 0 'succeeded' - meaning our injected exception did not "
          "actually fire as intended, or local[*] did not retry as expected. This would "
          "need investigating rather than assumed to match Databricks cluster behavior.")

# ---------------------------------------------------------------------------
# 5. Why this was cheap: narrow lineage here vs. what a wide-stage failure
#    would cost. Ties back to the ACTUAL toDebugString evidence from 3.1/3.2.
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 5: Why recovery was cheap here - narrow lineage - vs. a wide-stage failure")
print("=" * 80)

print("Narrow lineage (filtered_df) - toDebugString:")
print(lineage_str)
print(f"-> No ShuffledRowRDD present. Every partition here has exactly ONE parent "
      f"partition (the corresponding CSV file split) - a 1:1 narrow dependency. Losing "
      f"partition {TARGET_PARTITION} only required re-reading and re-filtering that ONE "
      f"file split, not the other {num_partitions - 1} partitions, and no shuffle.")

wide_rdd = filtered_df.groupBy("vehicle_id").agg(F.avg("engine_temp").alias("avg_engine_temp")).rdd
wide_debug_str = wide_rdd.toDebugString().decode("utf-8")
print("\nWide lineage (filtered_df.groupBy(vehicle_id).avg(engine_temp)) - toDebugString:")
print(wide_debug_str)
print(f"-> ShuffledRowRDD present - an N:1 wide dependency. An output partition here "
      f"depends on a SLICE of EVERY upstream partition (hash(vehicle_id) can route rows "
      f"from any of the {num_partitions} input partitions into any output partition). If "
      f"a post-shuffle partition were lost, its lineage says it must be rebuilt from the "
      f"shuffle's inputs - and if those were also gone, recomputation would cascade back "
      f"to re-reading and re-filtering ALL {num_partitions} upstream partitions, not just "
      f"one. This is exactly why lineage DEPTH matters: the deeper/wider the chain above a "
      f"lost partition, the more expensive recovery gets. Part 3.4 (checkpointing) exists "
      f"precisely to put a floor under this cost for long iterative chains, by "
      f"materializing intermediate state so recomputation doesn't have to walk all the "
      f"way back to the original source.")

# ---------------------------------------------------------------------------
# 6. This is NOT replication - contrast with Spark's opt-in persistence replication.
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("STEP 6: This is NOT replication - contrast with Spark's opt-in persistence replication")
print("=" * 80)
print(f"""
Nothing above involved Spark keeping a second copy of any partition's data.
By default, an RDD/DataFrame's intermediate results are NOT persisted at all
- if a partition is needed again, it is recomputed from lineage, exactly as
demonstrated above for partition {TARGET_PARTITION}.

Spark DOES offer an opt-in mechanism that looks superficially similar but is
architecturally different: .persist(StorageLevel.MEMORY_ONLY_2) (or
MEMORY_AND_DISK_2, etc.) asks Spark to cache a partition's data AND replicate
that cached copy to a second executor, so a lost partition can be re-fetched
from the replica instead of recomputed. That is:
  - opt-in, not default
  - a REPLICATION strategy (conceptually like Hadoop's 3x block replication,
    just typically 2x and in RAM rather than disk)
  - trading extra memory/network cost for faster recovery, worthwhile when
    recomputation would be more expensive than keeping a spare copy (very
    deep lineage, or expensive-to-recompute transformations)

We did not call .persist() anywhere in this script. The recovery observed
above for partition {TARGET_PARTITION} was 100% lineage-based recomputation - Spark re-ran
the Scan -> Project -> Filter chain for that one file split from the CSV
still sitting on disk. No replica was consulted because none exists.
""")

spark.stop()


spark.task.maxFailures = 4 (must be >1 for retry-on-failure to occur)
filtered_df partitions: 2 -> TARGET_PARTITION chosen as 1 (middle partition, computed from the real count above)

Lineage description: 6 lines / 341 bytes total - this fixed-size 'recipe' is ALL Spark needs to keep to recompute any partition.
Source CSV: 39,600,794 bytes / 500,000 rows -> ~79.2 bytes/row

[BASELINE - no failure injected] partition 1 row count: 221,262
[BASELINE] total row count across all 2 partitions: 495,000
Estimated data volume Spark must re-read/recompute if partition 1 is lost: ~17,524,302 bytes (~17113.6 KB)
Lineage-to-data leverage: 341 bytes of lineage metadata is enough to regenerate ~17113.6 KB of actual data - a 51,391x ratio. Spark stores the 341-byte recipe, not a redundant copy of the 17113.6 KB it produces.

STEP 4: Triggering job with a deliberate crash on partition 1, attempt 0.
Watch for a WARN 'Lost task 1.0 ...' line below, followed by the job completing anyway via a successful r

# F. Checkpointing

In [35]:
"""
Part 3.4 - Checkpointing to Truncate Excessive Lineage Depth
================================================================

FRESH SCRIPT - does not reuse any variable, DataFrame, or SparkContext from
3.1/3.2/3.3. Those may have been run in different sessions/processes; this
script re-reads the source CSV from scratch and builds everything itself.

TARGET ENVIRONMENT: Google Colab (per project decision - this script was
written but NOT executed by the assistant; run it in Colab and share the
real output back). Colab-specific choices below are called out explicitly
so porting to Databricks later just means swapping path strings, not logic.

WHY local-cluster[2,2,4096] instead of local[*]: Part 3.3 established
empirically that plain `local`/`local[*]` masters hardcode Spark's internal
maxTaskFailures to 1, ignoring spark.task.maxFailures entirely - so a
crashed task's first attempt aborts the whole job instead of being retried.
`local-cluster[N, cores, memory]` spawns genuine separate Worker/Executor
JVM processes and does NOT have that override, which is what Section 4
below (recovery-time comparison) needs to work at all.

Databricks portability note: sc.setCheckpointDir("/content/spark-checkpoints")
below is a Colab-local path; on Databricks the equivalent call is exactly
the same API against a DBFS path, e.g.
sc.setCheckpointDir("/dbfs/tmp/spark-checkpoints") - same mechanism, just a
different filesystem string.
"""

import os
import time
from pyspark import TaskContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# ---------------------------------------------------------------------------
# 0. Spark session (fresh - no reuse from earlier parts).
# ---------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("Part3_4_Checkpointing")
    .master("local-cluster[2,2,4096]")  # bumped from 1024MB - see driver-memory note below
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.task.maxFailures", "4")
    # Variant 3 crashed at the SAME iteration (15) with the SAME error even after the
    # executor-memory bump above - a strong signal the executors were never the
    # bottleneck. .checkpoint()/.toDebugString() do their plan-analysis work in the
    # DRIVER JVM, not the executors, so driver memory is a separate setting that needs
    # to be raised independently.
    # Budget note for Colab's free tier (~12GB instance): 2 workers x 4096MB + 4g driver
    # is already a meaningful chunk of that. If this still OOMs, reduce executor
    # count/memory first (e.g. local-cluster[2,2,2048]) rather than raising both
    # indefinitely.
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

CHECKPOINT_DIR = "/content/spark-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print(f"Checkpoint directory set to: {CHECKPOINT_DIR}")

DATA_PATH = "/content/synthetic_fleet_telemetry.csv"
schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_model", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("engine_temp", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("battery_efficiency", DoubleType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# ---------------------------------------------------------------------------
# 1. Ingest + narrow cleaning - rebuilt fresh, not imported from 3.1/3.2/3.3.
# ---------------------------------------------------------------------------
raw_df = spark.read.csv(DATA_PATH, header=True, schema=schema)
filtered_df = (
    raw_df.select("vehicle_id", "engine_temp")
    .filter(F.col("engine_temp").isNotNull() & (F.col("engine_temp") >= 60) & (F.col("engine_temp") <= 140))
)

# The iterative demo below re-derives its DataFrame from scratch on EVERY
# action, hundreds of times across 3 variants. Running that over the full
# ~495,000-row dataset would make this impractically slow without adding
# any evidentiary value - the "lineage becomes a liability" phenomenon is
# about CHAIN DEPTH, not row count. We deliberately use a small, fixed
# working set (5,000 rows) for this section only.
WORKING_ROWS = 5000
base_working_df = (
    filtered_df
    .limit(WORKING_ROWS)
    .withColumn("engine_temp_adj", F.col("engine_temp"))
)
base_working_df.cache()
actual_working_rows = base_working_df.count()
print(f"Working set for the iterative demo: {actual_working_rows:,} rows "
      f"(deliberately small - see comment above for why)")

NUM_ITERATIONS = 200
ACTION_EVERY = 10
CHECKPOINT_EVERY = 15


def apply_one_iteration(df, i):
    """
    One 'training step' - a stand-in for an iterative per-row update, e.g.
    a K-means centroid adjustment or a gradient-descent weight update
    against this telemetry data (Part 2.2's iterative-ML context). Each
    call wraps the PREVIOUS DataFrame in one more logical-plan node -
    that's what actually grows the lineage; the specific formula's
    numerical meaning doesn't matter for this demo.
    """
    return df.withColumn("engine_temp_adj", (F.col("engine_temp_adj") * 0.999) + F.lit(0.01))


def safe_lineage_length(df, label):
    """
    toDebugString() itself recursively walks the RDD DAG - if a variant's
    crash was a genuine StackOverflowError from deep recursion, calling
    toDebugString() on that same deep plan afterward could ALSO overflow.
    Guarded so a post-crash inspection failure doesn't kill the rest of
    this script's reporting.
    """
    try:
        return len(df.rdd.toDebugString())
    except Exception as e:
        print(f"  Could not measure lineage length for {label} - inspection itself failed: "
              f"{type(e).__name__}: {str(e)[:200]} (plausibly the same deep-recursion issue)")
        return None


# ---------------------------------------------------------------------------
# 2. VARIANT 1 - NO checkpointing, NO caching (the naive/broken case)
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print(f"VARIANT 1: NO checkpoint, NO cache - {NUM_ITERATIONS} iterations, "
      f"action every {ACTION_EVERY}")
print("=" * 80)

variant1_timings = []
variant1_crashed_at = None
variant1_crash_message = None

df_v1 = base_working_df
try:
    for i in range(1, NUM_ITERATIONS + 1):
        df_v1 = apply_one_iteration(df_v1, i)

        if i % ACTION_EVERY == 0 or i == NUM_ITERATIONS:
            t0 = time.perf_counter()
            cnt = df_v1.count()
            elapsed = time.perf_counter() - t0
            variant1_timings.append((i, elapsed, cnt))
            print(f"  [v1-none] iter {i:3d}: count()={cnt:,}, took {elapsed:.3f}s")
except Exception as e:
    variant1_crashed_at = i
    variant1_crash_message = f"{type(e).__name__}: {str(e)[:500]}"
    print(f"\n  *** VARIANT 1 CRASHED at iteration {i} ***")
    print(f"  Exception: {variant1_crash_message}")

variant1_final_lineage_len = safe_lineage_length(df_v1, "Variant 1 (none)")

if variant1_crashed_at is None:
    print(f"\nHONEST RESULT: Variant 1 completed all {NUM_ITERATIONS} iterations WITHOUT "
          f"crashing. The per-action timings printed above do NOT show a clear growth "
          f"trend at this scale (200 iterations, 5,000-row working set) - reporting that "
          f"plainly rather than reading a trend into noisy/flat numbers. Modern Spark's "
          f"Catalyst analyzer/optimizer has been hardened against deep recursive plans "
          f"compared to older versions, and wall-clock time alone doesn't reliably expose "
          f"the growing cost at this scale. The REAL evidence for lineage becoming a "
          f"liability here is the recorded lineage STRING LENGTH, not wall-clock time: "
          f"{variant1_final_lineage_len} characters after {NUM_ITERATIONS} iterations, "
          f"growing linearly with iteration count regardless of what the timings show - "
          f"that's what eventually risks the StackOverflow failure mode described in the "
          f"assignment at a large enough iteration count, even when per-action time stays flat.")
else:
    print(f"\nHONEST RESULT: Variant 1 DID crash at iteration {variant1_crashed_at}, "
          f"confirming the lineage-depth failure mode described in the assignment. "
          f"Lineage string length at the point of crash: "
          f"{variant1_final_lineage_len if variant1_final_lineage_len is not None else 'could not be measured - inspection itself failed, see message above'}.")

# ---------------------------------------------------------------------------
# 3. VARIANT 2 - CACHING ONLY, no checkpoint (proves caching alone doesn't fix this)
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print(f"VARIANT 2: CACHE only (no checkpoint) - {NUM_ITERATIONS} iterations, "
      f"action every {ACTION_EVERY}")
print("=" * 80)
print("Note: .cache() is lazy - marking a DataFrame for caching does nothing until an\n"
      "action runs against THAT specific DataFrame. Since we only force an action every\n"
      f"{ACTION_EVERY} iterations, most of the intermediate .cache() calls below never get\n"
      "materialized before being wrapped by the next iteration's transformation - this is\n"
      "realistic (and worth reporting honestly) rather than hidden.")

variant2_timings = []
variant2_crashed_at = None
variant2_crash_message = None

df_v2 = base_working_df
try:
    for i in range(1, NUM_ITERATIONS + 1):
        df_v2 = apply_one_iteration(df_v2, i).cache()

        if i % ACTION_EVERY == 0 or i == NUM_ITERATIONS:
            t0 = time.perf_counter()
            cnt = df_v2.count()
            elapsed = time.perf_counter() - t0
            variant2_timings.append((i, elapsed, cnt))
            print(f"  [v2-cache] iter {i:3d}: count()={cnt:,}, took {elapsed:.3f}s")
except Exception as e:
    variant2_crashed_at = i
    variant2_crash_message = f"{type(e).__name__}: {str(e)[:500]}"
    print(f"\n  *** VARIANT 2 CRASHED at iteration {i} ***")
    print(f"  Exception: {variant2_crash_message}")

variant2_final_lineage_len = safe_lineage_length(df_v2, "Variant 2 (cache)")

if variant2_crashed_at is None:
    print(f"\nHONEST RESULT: Variant 2 completed all {NUM_ITERATIONS} iterations too. "
          f"Compare its timings above to Variant 1's - if they track closely, that itself "
          f"is evidence that caching did not change the underlying lineage-growth cost.")
else:
    if variant1_crashed_at is None:
        comparison_clause = (
            f"BEFORE Variant 1 crashed - Variant 1 completed all {NUM_ITERATIONS} "
            f"iterations without crashing, so Variant 2 failed strictly sooner"
        )
    else:
        comparison_clause = f"Variant 1 crashed at iteration {variant1_crashed_at}, for comparison"
    print(f"\nHONEST RESULT: Variant 2 CRASHED at iteration {variant2_crashed_at} "
          f"({comparison_clause}). This is not simply 'caching didn't help' - it's a "
          f"specific mechanism: calling .cache() every iteration marks each intermediate "
          f"DataFrame for caching, but since we only force an action every {ACTION_EVERY} "
          f"iterations, most of those .cache() calls never get materialized before the "
          f"next iteration wraps them in a further transformation. Each unmaterialized "
          f".cache() still adds a node to Spark's internal plan/cache-metadata tracking on "
          f"top of the already-growing lineage, without ever delivering caching's actual "
          f"benefit (serving data from memory instead of recomputing) for those un-actioned "
          f"intermediate steps. The result is pure bookkeeping overhead stacked on top of "
          f"Variant 1's growth, not a mitigation of it.")

print(f"\nLineage string length at final/crash point - Variant 1 (none): "
      f"{variant1_final_lineage_len} chars, Variant 2 (cache): {variant2_final_lineage_len} chars")
print("If these two are close in size, it demonstrates caching does NOT shrink the "
      "recorded lineage - it only gives Spark the OPTION to serve data from memory "
      "instead of recomputing, without removing the plan history. If cached data is "
      "ever evicted, recovery still has to walk this same full-length chain back to "
      "the original source - exactly the risk checkpointing (Variant 3) removes.")

# ---------------------------------------------------------------------------
# 4. VARIANT 3 - CHECKPOINTING every 15 iterations (the fix)
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print(f"VARIANT 3: CHECKPOINT every {CHECKPOINT_EVERY} iterations - {NUM_ITERATIONS} "
      f"iterations, action every {ACTION_EVERY}")
print("=" * 80)

variant3_timings = []
variant3_crashed_at = None
variant3_crash_message = None
last_checkpoint_iteration = 0
truncation_proof_shown = 0

df_v3 = base_working_df
try:
    for i in range(1, NUM_ITERATIONS + 1):
        df_v3 = apply_one_iteration(df_v3, i)

        if i % CHECKPOINT_EVERY == 0:
            if truncation_proof_shown < 2:
                before_len = len(df_v3.rdd.toDebugString())
            df_v3 = df_v3.checkpoint(eager=True)  # eager=True: materializes NOW, truncates lineage immediately
            last_checkpoint_iteration = i
            if truncation_proof_shown < 2:
                after_len = len(df_v3.rdd.toDebugString())
                print(f"  [checkpoint @ iter {i}] toDebugString() length BEFORE checkpoint: "
                      f"{before_len} chars -> AFTER: {after_len} chars "
                      f"({'TRUNCATED - new leaf, no parent history' if after_len < before_len else 'NOT shorter - investigate'})")
                truncation_proof_shown += 1

        if i % ACTION_EVERY == 0 or i == NUM_ITERATIONS:
            t0 = time.perf_counter()
            cnt = df_v3.count()
            elapsed = time.perf_counter() - t0
            variant3_timings.append((i, elapsed, cnt))
            print(f"  [v3-checkpoint] iter {i:3d}: count()={cnt:,}, took {elapsed:.3f}s "
                  f"(iterations since last checkpoint: {i - last_checkpoint_iteration})")
except Exception as e:
    variant3_crashed_at = i
    variant3_crash_message = f"{type(e).__name__}: {str(e)[:500]}"
    print(f"\n  *** VARIANT 3 CRASHED at iteration {i} ***")
    print(f"  Exception: {variant3_crash_message}")

if variant3_crashed_at is None:
    variant3_final_lineage_len = safe_lineage_length(df_v3, "Variant 3 (checkpoint)")
    print(f"\nLineage string length at final iteration - Variant 3 (checkpoint): "
          f"{variant3_final_lineage_len} chars (only {NUM_ITERATIONS - last_checkpoint_iteration} "
          f"iterations of history since the last checkpoint at iteration {last_checkpoint_iteration}, "
          f"vs Variant 1/2's full {NUM_ITERATIONS}-iteration history)")
else:
    variant3_final_lineage_len = None

print("\n" + "-" * 80)
print("TIMING COMPARISON SUMMARY (per-action elapsed seconds, by iteration count)")
print("-" * 80)
print(f"{'iteration':<12}{'v1 (none)':<14}{'v2 (cache)':<14}{'v3 (checkpoint)':<16}")
v1_map = {i: t for i, t, _ in variant1_timings}
v2_map = {i: t for i, t, _ in variant2_timings}
v3_map = {i: t for i, t, _ in variant3_timings}
all_iters = sorted(set(v1_map) | set(v2_map) | set(v3_map))
for i in all_iters:
    v1s = f"{v1_map[i]:.3f}s" if i in v1_map else "n/a"
    v2s = f"{v2_map[i]:.3f}s" if i in v2_map else "n/a"
    v3s = f"{v3_map[i]:.3f}s" if i in v3_map else "n/a"
    print(f"{i:<12}{v1s:<14}{v2s:<14}{v3s:<16}")

# ---------------------------------------------------------------------------
# 5. RECOVERY-TIME COMPARISON - reuses the 3.3 TaskContext failure-injection
#    technique, once against the un-checkpointed chain and once against the
#    checkpointed chain, to make "stabilizes recovery time" concrete.
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 4: Recovery-time comparison - crash a partition near the end of each chain")
print("=" * 80)

recovery_results = {}

for label, df_chain, crashed_flag in [
    ("v1_no_checkpoint", df_v1, variant1_crashed_at),
    ("v3_checkpointed", df_v3, variant3_crashed_at),
]:
    if crashed_flag is not None:
        print(f"\nSkipping recovery-time test for {label}: this chain already crashed "
              f"during its own loop (at iteration {crashed_flag}), so there is no valid "
              f"final DataFrame to test recovery against.")
        continue

    try:
        num_partitions = df_chain.rdd.getNumPartitions()
        target_partition = num_partitions // 2

        def inject_failure_and_time(index, iterator, _target=target_partition):
            ctx = TaskContext.get()
            attempt = ctx.attemptNumber()
            rows = list(iterator)
            if index == _target and attempt == 0:
                raise RuntimeError(
                    f"INJECTED FAILURE ({label}): simulated crash on partition {index}, "
                    f"attempt {attempt}"
                )
            return iter([(index, len(rows), attempt)])

        # Baseline timing (no failure injected) for this same chain, to isolate
        # "cost added by the crash+retry specifically" from "cost inherent to
        # this chain's depth regardless of failure".
        t0 = time.perf_counter()
        _ = df_chain.rdd.mapPartitionsWithIndex(
            lambda idx, it: iter([(idx, sum(1 for _ in it), 0)])
        ).collect()
        baseline_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        result = df_chain.rdd.mapPartitionsWithIndex(inject_failure_and_time).collect()
        with_failure_time = time.perf_counter() - t0

        recovered = next(r for r in result if r[0] == target_partition)
        recovery_results[label] = {
            "baseline_time": baseline_time,
            "with_failure_time": with_failure_time,
            "recovery_overhead": with_failure_time - baseline_time,
            "recovered_attempt": recovered[2],
            "num_partitions": num_partitions,
            "target_partition": target_partition,
        }

        recovery_overhead = with_failure_time - baseline_time
        print(f"\n[{label}] partitions={num_partitions}, target_partition={target_partition}")
        print(f"  Baseline (no failure) time: {baseline_time:.3f}s")
        print(f"  With injected failure+retry time: {with_failure_time:.3f}s")
        print(f"  Recovery overhead (with_failure - baseline): {recovery_overhead:.3f}s")
        if recovery_overhead < 0:
            print(f"  NOTE: negative overhead is not a real finding - it means the 'with "
                  f"failure' run measured FASTER than the baseline run, which is impossible "
                  f"for what's actually being measured (retrying a crashed partition can "
                  f"only add work, never remove it). Flagging this as measurement noise "
                  f"(JVM/JIT warm-up variance between two separately-timed runs, or "
                  f"executor-scheduling jitter under local-cluster mode), not evidence of "
                  f"anything about checkpointing itself.")
        print(f"  Target partition succeeded on attempt: {recovered[2]} "
              f"({'retry occurred' if recovered[2] > 0 else 'NO retry occurred - unexpected, investigate'})")

    except Exception as e:
        print(f"\n  *** Recovery-time test for {label} raised an unexpected exception: "
              f"{type(e).__name__}: {str(e)[:500]} ***")
        print("  Reporting this honestly rather than hiding it - if this happens, the "
              "comparison below will be incomplete for this chain.")

if "v1_no_checkpoint" in recovery_results and "v3_checkpointed" in recovery_results:
    v1_overhead = recovery_results["v1_no_checkpoint"]["recovery_overhead"]
    v3_overhead = recovery_results["v3_checkpointed"]["recovery_overhead"]
    print(f"\n{'=' * 80}")
    print("RECOVERY OVERHEAD COMPARISON")
    print(f"{'=' * 80}")
    print(f"No-checkpoint chain ({NUM_ITERATIONS} iterations of history): "
          f"{v1_overhead:.3f}s recovery overhead")
    print(f"Checkpointed chain ({NUM_ITERATIONS - last_checkpoint_iteration} iterations "
          f"since last checkpoint): {v3_overhead:.3f}s recovery overhead")
    if v1_overhead < 0 or v3_overhead < 0:
        print(f"NOTE: at least one of these overhead numbers is negative (already flagged "
              f"above as measurement noise, not a real effect) - skipping the reduction-"
              f"factor claim below since it would be computed from noise, not signal. "
              f"Re-run with more iterations/rows if a clean comparison is needed for the report.")
    elif v3_overhead < v1_overhead:
        print(f"CONFIRMED: checkpointing reduced recovery overhead by "
              f"{v1_overhead - v3_overhead:.3f}s ({v1_overhead / max(v3_overhead, 1e-6):.1f}x), "
              f"because recomputing the lost partition only had to redo "
              f"{NUM_ITERATIONS - last_checkpoint_iteration} iterations of work, not "
              f"{NUM_ITERATIONS}.")
    else:
        print(f"UNEXPECTED: checkpointed recovery overhead ({v3_overhead:.3f}s) was NOT "
              f"lower than the uncheckpointed one ({v1_overhead:.3f}s). Reporting this "
              f"honestly - possible reasons: the working set (5,000 rows) may be too "
              f"small for recomputation cost to dominate over fixed task-scheduling "
              f"overhead, or JVM/JIT warm-up noise between the two separate measurements. "
              f"This would be worth re-running with more iterations/rows to confirm.")
else:
    print("\nRecovery-time comparison could not be completed for both chains (see skip/"
          "error messages above) - reporting incomplete rather than fabricating the "
          "missing side.")

# ---------------------------------------------------------------------------
# 6. Checkpointing vs caching - restated using THIS run's actual results
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SECTION 5: Checkpointing vs caching - what this run actually showed")
print("=" * 80)

if variant1_final_lineage_len is None or variant2_final_lineage_len is None:
    cache_vs_none_note = "could not be compared - lineage inspection itself failed for at least one variant (see message above)"
else:
    # Compare PER-ITERATION lineage growth, not raw totals - Variant 1 and Variant 2 can
    # reach different iteration counts (e.g. one crashes early), so a raw total-length
    # comparison would be misleading about which one's growth rate is actually worse.
    v1_iters_reached = variant1_crashed_at if variant1_crashed_at is not None else NUM_ITERATIONS
    v2_iters_reached = variant2_crashed_at if variant2_crashed_at is not None else NUM_ITERATIONS
    v1_chars_per_iter = variant1_final_lineage_len / max(v1_iters_reached, 1)
    v2_chars_per_iter = variant2_final_lineage_len / max(v2_iters_reached, 1)
    per_iter_ratio = v2_chars_per_iter / max(v1_chars_per_iter, 1e-9)

    if per_iter_ratio > 1.5:
        cache_vs_none_note = (
            f"Variant 2 carried {v2_chars_per_iter:.0f} chars/iteration of lineage vs "
            f"Variant 1's {v1_chars_per_iter:.0f} chars/iteration ({per_iter_ratio:.1f}x heavier "
            f"per step). This REINFORCES, not contradicts, the mechanism already identified for "
            f"Variant 2's crash: registering a DataFrame with Spark's cache manager on every "
            f"single iteration adds substantially more plan/metadata overhead per step than a "
            f"plain narrow withColumn(), even when most of those .cache() calls are never "
            f"materialized. Caching didn't just fail to shrink the lineage here - it made each "
            f"step of it heavier, which is consistent with Variant 2 failing at fewer iterations "
            f"than Variant 1 despite doing 'less work' on paper."
        )
    elif per_iter_ratio < 0.67:
        cache_vs_none_note = (
            f"Variant 2 carried {v2_chars_per_iter:.0f} chars/iteration vs Variant 1's "
            f"{v1_chars_per_iter:.0f} chars/iteration - Variant 2's per-step lineage was smaller, "
            f"which is the OPPOSITE of what the caching-overhead mechanism predicts. This is a "
            f"genuine anomaly worth re-examining rather than a clean confirmation, before "
            f"trusting either number for the report."
        )
    else:
        cache_vs_none_note = (
            f"comparable per-iteration lineage growth ({v1_chars_per_iter:.0f} vs "
            f"{v2_chars_per_iter:.0f} chars/iteration) - caching did NOT shrink the recorded "
            f"lineage, as theory predicts."
        )

if variant3_crashed_at or variant3_final_lineage_len is None:
    variant3_lineage_note = "skipped or unmeasurable - chain crashed or lineage inspection itself failed"
else:
    remaining_since_checkpoint = NUM_ITERATIONS - last_checkpoint_iteration
    variant3_lineage_note = (
        f"{variant3_final_lineage_len} chars - dramatically shorter, bounded by distance "
        f"since the last checkpoint ({remaining_since_checkpoint} iterations) rather than "
        f"growing with total iteration count"
    )

summary_text = f"""
From the project's technical checklist: .cache()/.persist() is a lazy,
in-memory (or disk) storage optimization that PRESERVES lineage - if the
cached data is evicted or was never materialized, Spark falls back to
recomputing from the original source via the full recorded chain.
.checkpoint() is an EAGER operation that materializes the RDD to reliable
storage (here, {CHECKPOINT_DIR}; DBFS on Databricks) and then discards its
list of parent dependencies - the checkpointed RDD becomes a new leaf node,
with no memory of how it was derived.

What this run actually measured:
  - Variant 1 (none) vs Variant 2 (cache) final lineage length: {variant1_final_lineage_len} vs {variant2_final_lineage_len} chars
    {cache_vs_none_note}
  - Variant 3 (checkpoint) final lineage length: {variant3_lineage_note}
  - Recovery overhead: see the RECOVERY OVERHEAD COMPARISON block above for
    the actual measured numbers from this run, not a theoretical claim.

Caching and checkpointing are solving DIFFERENT problems: caching trades
memory for avoiding redundant COMPUTATION during normal (non-failure)
execution; checkpointing trades one-time I/O cost for bounding RECOVERY
TIME and avoiding StackOverflow-class failures during PLANNING/ANALYSIS of
very deep chains. A production iterative job typically wants both: cache
for speed, checkpoint periodically for safety - they are complementary,
not substitutes for each other.
"""
print(summary_text)

spark.stop()


Checkpoint directory set to: /content/spark-checkpoints
Working set for the iterative demo: 5,000 rows (deliberately small - see comment above for why)

VARIANT 1: NO checkpoint, NO cache - 200 iterations, action every 10
  [v1-none] iter  10: count()=5,000, took 0.267s
  [v1-none] iter  20: count()=5,000, took 0.380s
  [v1-none] iter  30: count()=5,000, took 0.295s
  [v1-none] iter  40: count()=5,000, took 0.214s
  [v1-none] iter  50: count()=5,000, took 0.143s
  [v1-none] iter  60: count()=5,000, took 0.198s
  [v1-none] iter  70: count()=5,000, took 0.240s
  [v1-none] iter  80: count()=5,000, took 0.167s
  [v1-none] iter  90: count()=5,000, took 0.156s
  [v1-none] iter 100: count()=5,000, took 0.224s
  [v1-none] iter 110: count()=5,000, took 0.138s
  [v1-none] iter 120: count()=5,000, took 0.167s
  [v1-none] iter 130: count()=5,000, took 0.174s
  [v1-none] iter 140: count()=5,000, took 0.239s
  [v1-none] iter 150: count()=5,000, took 0.164s
  [v1-none] iter 160: count()=5,000, took 0